
# Transform Circuits Data

1. Read bronze circuits table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (circuitId -> circuit_id, circuitName -> circuit_name)
4. Rename columns to make them more meaningful (lat -> latitude, long -> longitude)
5. Filter out rows where circuit_id is null (business key validation)
6. Remove duplicate records
7. Transform values of columns circuit_name and locality to Title Case
8. Write the transformed data to silver circuits table

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../00-Common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"


### Step 1 - Read bronze circuits table

In [0]:
# circuits_df = spark.table(bronze_table)
circuits_df = spark.read.table(bronze_table)

### Step 2 - Keep only the columns required for analytics (Drop url column)



In [0]:
# circuits_selected_df = circuits_df.select(
#     col("cicrcuitId"),
#     col("cicrcuitName"),
#     col("lat"),
#     col("long"),
#     col("locality"),
#     col("country"),
#     col("ingestion_timestamp")
# )

In [0]:
circuits_selected_df = circuits_df.select(
    "cicrcuitId",
    "cicrcuitName",
    "lat",
    "long",
    "locality",
    "country",
    "ingestion_timestamp",
    "source_file"
)


### Step 3

- Standardise column names using snake_case (circuitId -> circuit_id, circuitName -> circuit_name)
- Rename columns to make them more meaningful (lat -> latitude, long -> longitude)

In [0]:
circuits_renamed_df = circuits_selected_df\
    .withColumnsRenamed(
        {
            "cicrcuitId":"circuit_id",
            "cicrcuitName":"circuit_name",
            "lat":"latitude",
            "long":"longitude"
        }
    )

In [0]:
# cicuits_renamed_df = circuits_selected_df\
#     .withColumnRenamed("cicrcuitId","circuit_id")\
#     .withColumnRenamed("cicrcuitName","circuit_name")\
#     .withColumnRenamed('lat',"latitude")\
#     .withColumnRenamed('long',"longitude")


### Step 5 - Filter out rows where circuit_id is null (business key validation)


In [0]:
# circuits_valid_df = circuits_renamed_df.filter(
#         "circuits_id IS NOT NULL"
#     )

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
        col("circuit_id").isNotNull()
    )


### Step  6 - Remove duplicate records

In [0]:
# circuits_valid_df.distinct() #applied on the entire record
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])


### Step  7 - Transform values of columns circuit_name and locality to Title Case

In [0]:
circuits_final_df = (
    circuits_distinct_df
    .withColumn('circuit_name', initcap('circuit_name'))
    .withColumn('locality',initcap("locality"))
)



### Step  8 - Write the transformed data to silver circuits table

In [0]:
(
    circuits_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(silver_table)
)